# 🔄 Базові робочі процеси агентів з моделями GitHub (Python)

## 📋 Навчальний посібник з оркестрації робочих процесів

Цей нотатник представляє потужні можливості **Workflow Builder** у Microsoft Agent Framework. Навчіться створювати складні багатоступеневі робочі процеси агентів, які можуть обробляти складні бізнес-процеси та безшовно координувати кілька AI-операцій.

## 🎯 Цілі навчання

### 🏗️ **Архітектура робочих процесів**
- **Workflow Builder**: Проєктування та оркестрація складних багатоступеневих процесів
- **Виконання на подіях**: Обробка подій робочого процесу та переходів станів
- **Візуальне проектування робочих процесів**: Створення та візуалізація структур робочих процесів
- **Інтеграція моделей GitHub**: Використання AI-моделей у контексті робочих процесів

### 🔄 **Оркестрація процесів**
- **Послідовні операції**: Поєднання кількох завдань агента у логічному порядку
- **Умовна логіка**: Впровадження точок прийняття рішень та розгалужень робочих процесів
- **Обробка помилок**: Надійне відновлення після помилок і стійкість робочих процесів
- **Управління станом**: Відстеження та керування станом виконання робочого процесу

### 📊 **Патерни робочих процесів для підприємств**
- **Автоматизація бізнес-процесів**: Автоматизація складних організаційних робочих процесів
- **Координація кількох агентів**: Координація кількох спеціалізованих агентів
- **Масштабоване виконання**: Проєктування робочих процесів для операцій масштабу підприємства
- **Моніторинг та спостереження**: Відстеження продуктивності та результатів робочих процесів

## ⚙️ Попередні вимоги та налаштування

### 📦 **Необхідні залежності**

Встановіть Agent Framework з можливостями робочих процесів:

```bash
pip install agent-framework-core -U
```

### 🔑 **Налаштування моделей GitHub**

**Налаштування середовища (.env файл):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Використання в підприємствах**

**Приклади бізнес-процесів:**
- **Реєстрація клієнтів**: Багатоступеневі робочі процеси перевірки та налаштування
- **Пайплайн контенту**: Автоматизоване створення, перегляд та публікація контенту
- **Обробка даних**: ETL-робочі процеси з AI-підсиленим перетворенням
- **Забезпечення якості**: Автоматизоване тестування та валідація

**Переваги робочих процесів:**
- 🎯 **Надійність**: Детерміноване виконання з відновленням після помилок
- 📈 **Масштабованість**: Обробка високого обсягу автоматизації процесів
- 🔍 **Спостереження**: Повні аудиторські журнали та моніторинг
- 🔧 **Обслуговуваність**: Візуальне проєктування та модульні компоненти

## 🎨 Патерни проєктування робочих процесів

### Базова структура робочого процесу
```mermaid
graph TD
    A[Початок] --> B[Завдання агента 1]
    B --> C{Точка прийняття рішення}
    C -->|Успіх| D[Завдання агента 2]
    C -->|Помилка| E[Обробник помилок]
    D --> F[Кінець]
    E --> F
```

**Ключові компоненти:**
- **WorkflowBuilder**: Головний рушій оркестрації
- **WorkflowEvent**: Обробка подій і комунікація
- **WorkflowViz**: Візуальне представлення робочого процесу та налагодження

Збудуємо ваш перший інтелектуальний робочий процес! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
